# Lab-2: Deep Feedforward Neural Network for Fashion MNIST Classification
## Reg.No: 2548514
### Load and Preprocess Fashion MNIST

In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [2]:
# Transform: convert images to tensors and normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


In [3]:
# Load training and test datasets
train_dataset = datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)


100%|██████████| 26.4M/26.4M [00:01<00:00, 21.2MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 340kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 6.24MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 7.40MB/s]


In [18]:
# DataLoader
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

### Construct a Deep Feedforward Neural Network (3+ Hidden Layers)

In [6]:
import torch.nn as nn

In [5]:
class DeepFeedForwardNN(nn.Module):
    def __init__(self):
        super(DeepFeedForwardNN, self).__init__()

        self.model = nn.Sequential(
            nn.Flatten(),           # 28x28 → 784
            nn.Linear(784, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)      # 10 classes
        )

    def forward(self, x):
        return self.model(x)


In [7]:
model = DeepFeedForwardNN()


### Define Loss Function and Optimizer (Cross-Entropy + Adam)

In [8]:
import torch.optim as optim

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

### 4. Training Loop (Forward Pass, Loss, Backward Pass, Update)

In [12]:
def train_model(model, train_loader, criterion, optimizer, epochs):
    for epoch in range(epochs):
        model.train()
        running_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            # Accuracy calculation
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss:.4f}, Accuracy: {accuracy:.2f}%")


In [15]:
epochs = 10
train_model(model, train_loader, criterion, optimizer, epochs)


Epoch [1/10], Loss: 470.9626, Accuracy: 81.65%
Epoch [2/10], Loss: 349.8004, Accuracy: 86.36%
Epoch [3/10], Loss: 312.1770, Accuracy: 87.63%
Epoch [4/10], Loss: 287.6501, Accuracy: 88.68%
Epoch [5/10], Loss: 272.3717, Accuracy: 89.33%
Epoch [6/10], Loss: 254.9385, Accuracy: 89.89%
Epoch [7/10], Loss: 241.4621, Accuracy: 90.32%
Epoch [8/10], Loss: 227.5759, Accuracy: 90.94%
Epoch [9/10], Loss: 217.5669, Accuracy: 91.26%
Epoch [10/10], Loss: 207.1365, Accuracy: 91.81%


### 5. Evaluate the Model on Unseen Test Data

In [19]:
def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    test_accuracy = 100 * correct / total
    print(f"Test Accuracy: {test_accuracy:.2f}%")


In [20]:
evaluate_model(model, test_loader)

Test Accuracy: 88.03%
